In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for Outbound FIA Table Schema and Data Update for DELETE_FLAG
# Purpose: Test and validate the cloning, schema transformation, and NOT NULL enforcement for DELETE_FLAG in 3 Delta tables
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script performs comprehensive PySpark-based tests for the Outbound FIA process. It clones three source tables, renames 'Filler_1' to 'DELETE_FLAG', enforces NOT NULL constraint, validates schema and data integrity, and covers error scenarios and data quality checks.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import DataFrame  
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StringType, IntegerType, LongType, FloatType, DoubleType, ShortType,
    DateType, TimestampType, StructType, StructField
)
from delta.tables import DeltaTable  

# --- Test configuration ---
CATALOG = "purgo_databricks"
SCHEMA = "purgo_playground"
SOURCE_TABLES = [
    "ods_free_drug",
    "ods_non_contracting",
    "ods_copay"
]
CLONE_TABLES = [
    "ods_free_drug_clone",
    "ods_non_contracting_clone",
    "ods_copay_clone"
]
# Mapping of source to clone
TABLE_MAP = dict(zip(SOURCE_TABLES, CLONE_TABLES))

# --- Utility Functions ---

def get_table_schema(table_full_name: str) -> StructType:
    """
    Get the schema of a table from the catalog.
    Args:
        table_full_name (str): Fully qualified table name (catalog.schema.table)
    Returns:
        StructType: PySpark schema object
    """
    # Purpose: Retrieve schema for validation and DDL generation
    return spark.table(table_full_name).schema

def validate_column_exists(schema: StructType, col_name: str) -> bool:
    """
    Check if a column exists in the schema.
    Args:
        schema (StructType): Table schema
        col_name (str): Column name to check
    Returns:
        bool: True if exists, False otherwise
    """
    # Purpose: Ensure required columns are present
    return col_name in [f.name for f in schema.fields]

def validate_column_type(schema: StructType, col_name: str, expected_type: type) -> bool:
    """
    Validate the type of a column in the schema.
    Args:
        schema (StructType): Table schema
        col_name (str): Column name
        expected_type (type): Expected PySpark type
    Returns:
        bool: True if type matches, False otherwise
    """
    # Purpose: Ensure column types are correct
    for f in schema.fields:
        if f.name == col_name:
            return isinstance(f.dataType, expected_type)
    return False

def drop_table_if_exists(table_full_name: str):
    """
    Drop a table if it exists.
    Args:
        table_full_name (str): Fully qualified table name
    Returns:
        None
    """
    # Purpose: Cleanup before clone creation
    spark.sql(f"DROP TABLE IF EXISTS {table_full_name}")

def create_delta_table_with_not_null(table_name: str, schema: StructType, not_null_col: str):
    """
    Create a Delta table with NOT NULL constraint on specified column.
    Args:
        table_name (str): Fully qualified table name
        schema (StructType): Table schema
        not_null_col (str): Column to enforce NOT NULL
    Returns:
        None
    """
    # Purpose: Create table with correct constraints
    cols_ddl = []
    for f in schema.fields:
        col_type = None
        if isinstance(f.dataType, StringType):
            col_type = "STRING"
        elif isinstance(f.dataType, LongType):
            col_type = "BIGINT"
        elif isinstance(f.dataType, IntegerType):
            col_type = "INT"
        elif isinstance(f.dataType, FloatType):
            col_type = "FLOAT"
        elif isinstance(f.dataType, DoubleType):
            col_type = "DOUBLE"
        elif isinstance(f.dataType, ShortType):
            col_type = "SMALLINT"
        elif isinstance(f.dataType, DateType):
            col_type = "DATE"
        elif isinstance(f.dataType, TimestampType):
            col_type = "TIMESTAMP"
        else:
            raise Exception(f"Unsupported type for column {f.name}: {f.dataType}")
        null_str = "NOT NULL" if f.name == not_null_col else ""
        cols_ddl.append(f"{f.name} {col_type} {null_str}".strip())
    # Add constraint for NOT NULL
    constraint_ddl = f"CONSTRAINT {not_null_col}_not_null CHECK ({not_null_col} IS NOT NULL)"
    ddl = f"""
    CREATE TABLE {table_name} (
        {', '.join(cols_ddl)},
        {constraint_ddl}
    ) USING DELTA
    """
    spark.sql(ddl)

def clone_table_with_delete_flag(source_table: str, clone_table: str):
    """
    Clone a source table to a clone table, renaming Filler_1 to DELETE_FLAG and enforcing NOT NULL.
    Args:
        source_table (str): Source table name (without catalog/schema)
        clone_table (str): Clone table name (without catalog/schema)
    Returns:
        None
    """
    """
    Steps:
    - Drop clone table if exists
    - Read source table
    - Validate Filler_1 exists and is string
    - Filter rows where Filler_1 IS NOT NULL
    - Rename Filler_1 to DELETE_FLAG
    - Create clone table with NOT NULL constraint on DELETE_FLAG
    - Insert data
    """
    full_source = f"{CATALOG}.{SCHEMA}.{source_table}"
    full_clone = f"{CATALOG}.{SCHEMA}.{clone_table}"
    drop_table_if_exists(full_clone)
    try:
        df = spark.table(full_source)
    except Exception as e:
        raise Exception(f"Source table {source_table} does not exist")
    schema = df.schema
    if not validate_column_exists(schema, "Filler_1"):
        raise Exception(f"Source table {source_table} missing required column 'Filler_1'")
    if not validate_column_type(schema, "Filler_1", StringType):
        actual_type = [f.dataType for f in schema.fields if f.name == "Filler_1"][0]
        raise Exception(f"Column 'Filler_1' in {source_table} must be of type string, found {actual_type}")
    # Filter rows where Filler_1 IS NOT NULL
    df_filtered = df.filter(F.col("Filler_1").isNotNull())
    # Rename Filler_1 to DELETE_FLAG
    cols = [f.name for f in schema.fields]
    new_cols = []
    for c in cols:
        if c == "Filler_1":
            new_cols.append(F.col("Filler_1").alias("DELETE_FLAG"))
        else:
            new_cols.append(F.col(c))
    df_renamed = df_filtered.select(*new_cols)
    # Build new schema: replace Filler_1 with DELETE_FLAG, enforce NOT NULL
    new_fields = []
    for f in schema.fields:
        if f.name == "Filler_1":
            new_fields.append(StructField("DELETE_FLAG", StringType(), False))
        else:
            new_fields.append(f)
    new_schema = StructType(new_fields)
    # Create clone table with NOT NULL constraint
    create_delta_table_with_not_null(full_clone, new_schema, "DELETE_FLAG")
    # Insert data
    df_renamed.write.format("delta").mode("append").saveAsTable(full_clone)

def assert_table_schema(table_full_name: str, expected_schema: StructType, not_null_col: str):
    """
    Assert that a table's schema matches the expected schema and NOT NULL constraint.
    Args:
        table_full_name (str): Fully qualified table name
        expected_schema (StructType): Expected schema
        not_null_col (str): Column to check for NOT NULL
    Returns:
        None
    """
    # Purpose: Validate schema and constraints
    actual_schema = get_table_schema(table_full_name)
    assert len(actual_schema.fields) == len(expected_schema.fields), f"Column count mismatch in {table_full_name}"
    for ef, af in zip(expected_schema.fields, actual_schema.fields):
        assert ef.name == af.name, f"Column name mismatch: {ef.name} != {af.name}"
        assert type(ef.dataType) == type(af.dataType), f"Type mismatch for {ef.name}: {ef.dataType} != {af.dataType}"
        if ef.name == not_null_col:
            assert af.nullable is False, f"{not_null_col} column must be NOT NULL in {table_full_name}"

def assert_no_nulls_in_column(table_full_name: str, col_name: str):
    """
    Assert that a column contains no NULL values.
    Args:
        table_full_name (str): Fully qualified table name
        col_name (str): Column name
    Returns:
        None
    """
    # Purpose: Data quality validation
    df = spark.table(table_full_name)
    null_count = df.filter(F.col(col_name).isNull()).count()
    assert null_count == 0, f"NOT NULL constraint failed for column '{col_name}' in {table_full_name}"

def assert_data_integrity(source_table: str, clone_table: str):
    """
    Assert that all rows in clone table have DELETE_FLAG not null and match source table (except Filler_1 renamed).
    Args:
        source_table (str): Source table name
        clone_table (str): Clone table name
    Returns:
        None
    """
    # Purpose: End-to-end data validation
    full_source = f"{CATALOG}.{SCHEMA}.{source_table}"
    full_clone = f"{CATALOG}.{SCHEMA}.{clone_table}"
    src_df = spark.table(full_source).filter(F.col("Filler_1").isNotNull())
    clone_df = spark.table(full_clone)
    # Rename Filler_1 to DELETE_FLAG in source for comparison
    src_cols = [c for c in src_df.columns if c != "Filler_1"] + ["DELETE_FLAG"]
    src_df_renamed = src_df.withColumnRenamed("Filler_1", "DELETE_FLAG")
    # Reorder columns for comparison
    src_df_renamed = src_df_renamed.select(*clone_df.columns)
    # Compare row counts
    assert src_df_renamed.count() == clone_df.count(), f"Row count mismatch between {source_table} and {clone_table}"
    # Compare data
    src_rows = set([tuple(row) for row in src_df_renamed.collect()])
    clone_rows = set([tuple(row) for row in clone_df.collect()])
    assert src_rows == clone_rows, f"Data mismatch between {source_table} and {clone_table}"

def assert_insert_null_fails(table_full_name: str, schema: StructType, not_null_col: str):
    """
    Assert that inserting a row with NULL in NOT NULL column fails.
    Args:
        table_full_name (str): Fully qualified table name
        schema (StructType): Table schema
        not_null_col (str): NOT NULL column
    Returns:
        None
    """
    # Purpose: Constraint enforcement test
    row = []
    for f in schema.fields:
        if f.name == not_null_col:
            row.append(None)
        else:
            # Use dummy value for other columns
            if isinstance(f.dataType, StringType):
                row.append("dummy")
            elif isinstance(f.dataType, LongType):
                row.append(1)
            elif isinstance(f.dataType, IntegerType):
                row.append(1)
            elif isinstance(f.dataType, FloatType):
                row.append(1.0)
            elif isinstance(f.dataType, DoubleType):
                row.append(1.0)
            elif isinstance(f.dataType, ShortType):
                row.append(1)
            elif isinstance(f.dataType, DateType):
                row.append("2024-01-01")
            elif isinstance(f.dataType, TimestampType):
                row.append("2024-01-01 00:00:00")
            else:
                row.append("dummy")
    df = spark.createDataFrame([tuple(row)], schema)
    try:
        df.write.format("delta").mode("append").saveAsTable(table_full_name)
        raise Exception(f"NOT NULL constraint failed for column '{not_null_col}' in {table_full_name}")
    except Exception as e:
        assert "NOT NULL" in str(e) or "constraint" in str(e), f"Expected NOT NULL constraint error, got: {str(e)}"

def assert_extra_column_ignored(source_table: str, clone_table: str, extra_col: str):
    """
    Assert that extra columns in source table are ignored in clone table.
    Args:
        source_table (str): Source table name
        clone_table (str): Clone table name
        extra_col (str): Extra column name
    Returns:
        None
    """
    # Purpose: Schema consistency test
    full_clone = f"{CATALOG}.{SCHEMA}.{clone_table}"
    clone_schema = get_table_schema(full_clone)
    assert extra_col not in [f.name for f in clone_schema.fields], f"Extra column {extra_col} should not exist in {clone_table}"

def assert_duplicate_rows_cloned(source_table: str, clone_table: str):
    """
    Assert that duplicate rows in source table are cloned to clone table.
    Args:
        source_table (str): Source table name
        clone_table (str): Clone table name
    Returns:
        None
    """
    # Purpose: Data duplication test
    full_source = f"{CATALOG}.{SCHEMA}.{source_table}"
    full_clone = f"{CATALOG}.{SCHEMA}.{clone_table}"
    src_df = spark.table(full_source).filter(F.col("Filler_1").isNotNull())
    clone_df = spark.table(full_clone)
    # Count duplicates in source and clone
    src_counts = src_df.groupBy(src_df.columns).count().filter(F.col("count") > 1).count()
    clone_counts = clone_df.groupBy(clone_df.columns).count().filter(F.col("count") > 1).count()
    assert src_counts == clone_counts, f"Duplicate row count mismatch between {source_table} and {clone_table}"

def assert_missing_required_column_fails(source_table: str, missing_col: str):
    """
    Assert that missing required column in source table raises error.
    Args:
        source_table (str): Source table name
        missing_col (str): Missing column name
    Returns:
        None
    """
    # Purpose: Error handling test
    full_source = f"{CATALOG}.{SCHEMA}.{source_table}"
    try:
        schema = get_table_schema(full_source)
        if not validate_column_exists(schema, missing_col):
            raise Exception(f"Source table {source_table} missing required column '{missing_col}'")
    except Exception as e:
        assert missing_col in str(e), f"Expected missing column error for {missing_col}, got: {str(e)}"

def assert_column_type_mismatch_fails(source_table: str, col: str, expected_type: type):
    """
    Assert that column type mismatch raises error.
    Args:
        source_table (str): Source table name
        col (str): Column name
        expected_type (type): Expected PySpark type
    Returns:
        None
    """
    # Purpose: Error handling test
    full_source = f"{CATALOG}.{SCHEMA}.{source_table}"
    schema = get_table_schema(full_source)
    if not validate_column_type(schema, col, expected_type):
        actual_type = [f.dataType for f in schema.fields if f.name == col][0]
        raise Exception(f"Column '{col}' in {source_table} must be of type {expected_type.__name__}, found {actual_type}")

def assert_clone_table_dropped_before_creation(clone_table: str):
    """
    Assert that clone table is dropped before creation.
    Args:
        clone_table (str): Clone table name
    Returns:
        None
    """
    # Purpose: Table lifecycle test
    full_clone = f"{CATALOG}.{SCHEMA}.{clone_table}"
    drop_table_if_exists(full_clone)
    # Table should not exist now
    try:
        spark.table(full_clone)
        raise Exception(f"Clone table {clone_table} should have been dropped")
    except Exception:
        pass  # Expected

def assert_delete_flag_nullable_fails(clone_table: str):
    """
    Assert that DELETE_FLAG column defined as nullable raises error.
    Args:
        clone_table (str): Clone table name
    Returns:
        None
    """
    # Purpose: Constraint enforcement test
    full_clone = f"{CATALOG}.{SCHEMA}.{clone_table}"
    schema = get_table_schema(full_clone)
    for f in schema.fields:
        if f.name == "DELETE_FLAG":
            if f.nullable:
                raise Exception(f"DELETE_FLAG column must be defined as NOT NULL in {clone_table}")

def assert_clone_from_nonexistent_source_fails(source_table: str):
    """
    Assert that cloning from non-existent source table raises error.
    Args:
        source_table (str): Source table name
    Returns:
        None
    """
    # Purpose: Error handling test
    full_source = f"{CATALOG}.{SCHEMA}.{source_table}"
    try:
        spark.table(full_source)
        raise Exception(f"Source table {source_table} does not exist")
    except Exception as e:
        assert "does not exist" in str(e) or "not found" in str(e), f"Expected not found error, got: {str(e)}"

# --- Main Test Execution ---

def run_all_tests():
    """
    Run all test cases for Outbound FIA Table Schema and Data Update for DELETE_FLAG.
    Args:
        None
    Returns:
        None
    """
    # Purpose: Orchestrate all test scenarios
    for src, clone in TABLE_MAP.items():
        # Clone table creation and validation
        clone_table_with_delete_flag(src, clone)
        # Validate schema: DELETE_FLAG is NOT NULL
        src_schema = get_table_schema(f"{CATALOG}.{SCHEMA}.{src}")
        # Build expected schema
        expected_fields = []
        for f in src_schema.fields:
            if f.name == "Filler_1":
                expected_fields.append(StructField("DELETE_FLAG", StringType(), False))
            else:
                expected_fields.append(f)
        expected_schema = StructType(expected_fields)
        assert_table_schema(f"{CATALOG}.{SCHEMA}.{clone}", expected_schema, "DELETE_FLAG")
        # Validate data: no NULLs in DELETE_FLAG
        assert_no_nulls_in_column(f"{CATALOG}.{SCHEMA}.{clone}", "DELETE_FLAG")
        # Validate data integrity
        assert_data_integrity(src, clone)
        # Test NOT NULL constraint enforcement
        assert_insert_null_fails(f"{CATALOG}.{SCHEMA}.{clone}", expected_schema, "DELETE_FLAG")
        # Test extra column ignored (simulate by checking schema)
        # For demonstration, use a dummy extra column name
        assert_extra_column_ignored(src, clone, "Extra_Column_1")
        # Test duplicate rows cloned
        assert_duplicate_rows_cloned(src, clone)
        # Test missing required column
        assert_missing_required_column_fails(src, "Drug_Name" if src == "ods_free_drug" else ("Facility" if src == "ods_non_contracting" else "NDC"))
        # Test column type mismatch
        if src == "ods_free_drug":
            assert_column_type_mismatch_fails(src, "Quantity", LongType)
        elif src == "ods_non_contracting":
            assert_column_type_mismatch_fails(src, "Units_Provided", LongType)
        elif src == "ods_copay":
            assert_column_type_mismatch_fails(src, "Copay_Amount", LongType)
        # Test clone table dropped before creation
        assert_clone_table_dropped_before_creation(clone)
        # Test DELETE_FLAG nullable fails
        assert_delete_flag_nullable_fails(clone)
        # Test clone from non-existent source table
        assert_clone_from_nonexistent_source_fails("nonexistent_table")

# Run all tests
run_all_tests()

# spark.stop()  # Do not stop SparkSession in Databricks
